# Этап 2. Подготовка датасета

### Автор: Балицкая Ксения

In [49]:
import json
import pandas as pd
from pathlib import Path
import numpy as np
import ast

In [2]:
# Путь до наших сгенерированных данных в папке Fashion_CV/create_dataset/
DATA_PATH = Path('../create_dataset/data/dataset_annotations.json')

In [3]:
if not DATA_PATH.exists():
    print(f"Файл не найден по пути: {DATA_PATH.resolve()}")
else:
    with open(DATA_PATH, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
    
    print(f"Успешно загружено {len(raw_data)} записей")

Успешно загружено 15000 записей


#### 1. Преобразование JSON в DataFrame

In [12]:
df_raw = pd.DataFrame.from_dict(raw_data, orient='index')
df_raw.head(2)

,plaid_shirt_dress,grey_sweater,neon_beanie,black_trousers,black_boots,ankle_boot_socks,black_jacket,neon_yellow_beanie,brown_bag,black_leather_jacket,...,blue_rings,brown_handbag,grey_and_white_striped_socks,black_ankle_strap_socks,white_pants,black_high_waisted_jean_pants,white_collar_shirt,chunky_scarf,scuffed_ankle_boots,brown belt
images_for_training/Paisley_Rose_Print_Dress_img_00000024.jpg,"{'item_type': 'shirt dress', 'demographic': 'w...","{'item_type': 'sweater', 'demographic': 'unise...","{'item_type': 'beanie', 'demographic': 'unisex...","{'item_type': 'trousers', 'demographic': 'unis...","{'item_type': 'boots', 'demographic': 'unisex'...","{'item_type': 'socks', 'demographic': 'unisex'...",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
images_for_training/WOMEN1_Tees_Tanks_id_00004889_06_4_full.jpg,"{'item_type': 'shirt dress', 'demographic': 'w...","{'item_type': 'sweater', 'demographic': 'unise...","{'item_type': 'beanie', 'demographic': 'unisex...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
df_stacked = df_raw.stack().reset_index()
df_stacked.columns = ['image_path', 'item_name', 'attributes']
df_attributes = pd.json_normalize(df_stacked['attributes'])

In [14]:
df = pd.concat([df_stacked[['image_path', 'item_name']], df_attributes], axis=1)
df.head(2)

,image_path,item_name,item_type,demographic,age_group,color,silhouette,design_features,length,volume,functional_elements,decorative_elements,accessories,season,purpose,style,usage_conditions
0,images_for_training/Paisley_Rose_Print_Dress_i...,plaid_shirt_dress,shirt dress,women,young adults 18-35,black with white grid,fitted,"button-down, V-neck",knee-length,tight,"[buttons, pockets]",[grid pattern],"[scarf, ankle boots]",fall,casual,grunge,"outdoor, casual walk"
1,images_for_training/Paisley_Rose_Print_Dress_i...,grey_sweater,sweater,unisex,all ages,heather grey,oversized,ribbed knit,None,loose,[pockets],[chunky knit],"[neon beanie, sunglasses]",winter,casual,minimalist,outdoor


In [16]:
# Размер датасета
df.shape

(62979, 17)

#### 2. Очистка данных

In [17]:
df.describe()

,image_path,item_name,item_type,demographic,age_group,color,silhouette,design_features,length,volume,functional_elements,decorative_elements,accessories,season,purpose,style,usage_conditions
count,62979,62979,62979,62979,62979,62979,54335,60226,45377,53968,61717,61066,60691,62949,62971,62966,62971
unique,15000,363,76,3,3,65,69,1219,26,5,493,405,460,7,27,44,38
top,images_for_training/Socialite_Halter_Dress_img...,plaid_shirt_dress,sweater,unisex,all ages,neon yellow,fitted,[ribbed knit],null,loose,[],[],[],all-season,casual,minimalist,outdoor
freq,18,14691,14954,41624,44838,14940,20493,14750,23841,27149,19125,20044,29195,26793,55697,24811,38522


In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62979 entries, 0 to 62978
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   image_path           62979 non-null  object
 1   item_name            62979 non-null  object
 2   item_type            62979 non-null  object
 3   demographic          62979 non-null  object
 4   age_group            62979 non-null  object
 5   color                62979 non-null  object
 6   silhouette           54335 non-null  object
 7   design_features      60226 non-null  object
 8   length               45377 non-null  object
 9   volume               53968 non-null  object
 10  functional_elements  61717 non-null  object
 11  decorative_elements  61066 non-null  object
 12  accessories          60691 non-null  object
 13  season               62949 non-null  object
 14  purpose              62971 non-null  object
 15  style                62966 non-null  object
 16  usag

Как мы видим, есть пропущенные значения, которые достаточно часто встречаются в некоторых колонках

In [27]:
for name in df.columns:
    if df[name].isna().sum() != 0:
        print(f"В колонке {name} - {df[name].isna().sum()} пропущенных значений")

В колонке silhouette - 8644 пропущенных значений
В колонке design_features - 2753 пропущенных значений
В колонке length - 17602 пропущенных значений
В колонке volume - 9011 пропущенных значений
В колонке functional_elements - 1262 пропущенных значений
В колонке decorative_elements - 1913 пропущенных значений
В колонке accessories - 2288 пропущенных значений
В колонке season - 30 пропущенных значений
В колонке purpose - 8 пропущенных значений
В колонке style - 13 пропущенных значений
В колонке usage_conditions - 8 пропущенных значений


Распишу стратегию работы с фотографиями:
1. У нас есть колонка `image_path`, которая не несет в себе никакой информации, помимо пути к нашим фотографиям. Эту колонку можно было бы удалить, но возможно, что на 3-м этапе она понадобится, когда будет обучение нейронной сети, поэтому эту колонку можно просто проигнорировать при подаче признаков в сеть
2. Также у нас есть проблемные колонки в которых отстуствует большая часть информации:
    * `silhouette` - силуэт (8644 пропуска)
    * `design_features` - детали дизайна (2753 пропуска)
    * `length` - длина (17602 пропуска)
    * `volume` - объем (9011 пропусков)
    * `functional_elements`/`decorative_elements` - карманы, молнии, рюши, принты (вместе 3175 пропусков)
    * `accessories` - наличие ремня, очков, сумки (2288 пропусков) \

    Для базового подбора одежды данные признаки можно удалить, а сотавить только `design_features`, заполнив NaN пустым списком [] т.к. это наиболее важная колонка
3. Остальные пропуски можно заполнить словом 'unknown' т.к. мы работаем не просто с данными, они должны соотвествовать фотографиям.

In [28]:
# Удаление колонок 
columns_to_drop = ["silhouette", "length", "volume", "functional_elements", "decorative_elements", "accessories"]

df = df.drop(columns=columns_to_drop)
df.shape

(62979, 11)

In [40]:
# Работа с колонкой design_features
df['design_features'].nunique

<bound method IndexOpsMixin.nunique of 0           button-down, V-neck
1                   ribbed knit
2                          None
3                      slim fit
4                          None
                  ...          
62974               ribbed knit
62975     [button-down, V-neck]
62976    [lacing, treaded sole]
62977             [ribbed knit]
62978             [ribbed knit]
Name: design_features, Length: 62979, dtype: object>

В этой колонке у нас есть не только пустые значения, но и нормальный список, или строка через запятую, или строка, которая выглядит, как список. Все это нужно привести к единому виду

In [43]:
def clean_list_column(x):
    # 1. Если это пустота 
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    
    # 2. Если это уже нормальный питоновский список
    if isinstance(x, list):
        return x
    
    # 3. Если это строка, которая выглядит как список 
    if isinstance(x, str) and x.startswith('['):
        try:
            return ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return []
            
    # 4. Если это просто строка через запятую 
    if isinstance(x, str):
        return [item.strip() for item in x.split(',') if item.strip()]
        
    return []

In [44]:
df['design_features'] = df['design_features'].apply(clean_list_column)

In [47]:
# Остальные пропуски заполняем словом 'unknown'
df = df.fillna('unknown')
df.head()

,image_path,item_name,item_type,demographic,age_group,color,design_features,season,purpose,style,usage_conditions
0,images_for_training/Paisley_Rose_Print_Dress_i...,plaid_shirt_dress,shirt dress,women,young adults 18-35,black with white grid,"[button-down, V-neck]",fall,casual,grunge,"outdoor, casual walk"
1,images_for_training/Paisley_Rose_Print_Dress_i...,grey_sweater,sweater,unisex,all ages,heather grey,[ribbed knit],winter,casual,minimalist,outdoor
2,images_for_training/Paisley_Rose_Print_Dress_i...,neon_beanie,beanie,unisex,all ages,neon yellow,[],unknown,casual,street style,unknown
3,images_for_training/Paisley_Rose_Print_Dress_i...,black_trousers,trousers,unisex,all ages,black,[slim fit],all-season,casual,minimalist,"outdoor, casual walk"
4,images_for_training/Paisley_Rose_Print_Dress_i...,black_boots,boots,unisex,all ages,black,[],winter,casual,grunge,outdoor


Для колонки `usage_conditions` тоже сделаем список в значениях, применив функцию выше

In [48]:
df['usage_conditions'] = df['usage_conditions'].apply(clean_list_column)
df.head()

,image_path,item_name,item_type,demographic,age_group,color,design_features,season,purpose,style,usage_conditions
0,images_for_training/Paisley_Rose_Print_Dress_i...,plaid_shirt_dress,shirt dress,women,young adults 18-35,black with white grid,"[button-down, V-neck]",fall,casual,grunge,"[outdoor, casual walk]"
1,images_for_training/Paisley_Rose_Print_Dress_i...,grey_sweater,sweater,unisex,all ages,heather grey,[ribbed knit],winter,casual,minimalist,[outdoor]
2,images_for_training/Paisley_Rose_Print_Dress_i...,neon_beanie,beanie,unisex,all ages,neon yellow,[],unknown,casual,street style,[unknown]
3,images_for_training/Paisley_Rose_Print_Dress_i...,black_trousers,trousers,unisex,all ages,black,[slim fit],all-season,casual,minimalist,"[outdoor, casual walk]"
4,images_for_training/Paisley_Rose_Print_Dress_i...,black_boots,boots,unisex,all ages,black,[],winter,casual,grunge,[outdoor]


In [52]:
df.describe()

,image_path,item_name,item_type,demographic,age_group,color,design_features,season,purpose,style,usage_conditions
count,62979,62979,62979,62979,62979,62979,62979,62979,62979,62979,62979
unique,15000,363,76,3,3,65,1041,8,28,45,39
top,images_for_training/Socialite_Halter_Dress_img...,plaid_shirt_dress,sweater,unisex,all ages,neon yellow,[ribbed knit],all-season,casual,minimalist,[outdoor]
freq,18,14691,14954,41624,44838,14940,16718,26793,55697,24811,38522


#### 3. Кодирование признаков

Нам нужно закодировать все колонки, кроме `image_path` т.к. там лежат пути до фотографий. Для оставшихся колонок действуем следующим образом:
1. `item_type`, `season`, `purpose`, `style` - Скорей всего тут есть значения, которые встречаются 1-2 раза, поэтому можно сгруппировать редкие классы в 'other', а остальные закодировать через LabelEncoder или OHE.
2. `demographic`, `age_group` - Тут можно использовать кодировку OHE сразу т.к. всего 3 уникальных значения
3. `color` - как можно заметить выше, нейросеть писала в json сложные цвета (например, neon yellow или heather grey). Можно попробовать написать простую функцию, которая группирует сложные цвета и приводит их к простым (yellow, grey и т.д.)
4. `design_features`- 1041 уникальное значение. Поскольку здесь лежат списки деталей дизайна, можно попробовать использовать MultiLabelBinarizer. Но чтобы не плодить 1041 колонку, можно выбрать только самые популярные
5. `item_name` - по сути эта колонка дублирует `item_type` и `color`, поэтому она не нужна и ее можно просто удалить.
6. `usage_conditions` - эта колонка тоже дублирует информацию из `purpose` и `style` + по пользовательским фотографиям сложно будет определить, например, одежду для прогулки от одежды для похода в кино, поэтому эта колонка будет беспощадно удалена

#### 4. Сохранение датасета для ML